# 📊 Clinisys Embrioes Metrics: DuckDB vs AWS Athena

This notebook connects to both the local **DuckDB** database (`huntington_data_lake.duckdb`) and **AWS Athena (Production)** (`gold_huntington_prod`) to execute data quality and validation queries on the `clinisys_embrioes` table.

### Key Metrics Reconciled:
1. **Total Row Count (`total_rows`)**
2. **Unique Oocito IDs (`unique_oocito_id`) & Duplicate Check**
3. **Presence of Linked Entity IDs**:
   - `trat1_id` (Primary Treatment)
   - `emb_cong_id` (Frozen Embryo)
   - `cong_em_id` (Freezing Procedure)
   - `descong_em_id` (Thawing Procedure)
   - `trat2_id` (Transfer Treatment)

### Breakdowns & Analyses Included:
- **Part 1 & 2**: DuckDB & Athena Overall, Per Year (`micro_data_procedimento`), and Per Unidade (`patient_info.unidade_nome`)
- **Part 3**: Side-by-Side Reconciliation Summary Table
- **Part 4**: Discrepancy Analysis — Oocito IDs present ONLY in DuckDB or ONLY in Athena
- **Part 5**: Attribute Discrepancy Analysis — Per-key comparison (`trat1_id`, `emb_cong_id`, `cong_em_id`, `descong_em_id`, `trat2_id`, `oocito_embryo_number`) for matching oocitos
- **Part 6**: PGT / Genetic Test Results Side-by-Side Comparison (`oocito_resultado_pgd`, normalized NULLs/empty)
- **Part 7**: Date Sequence & Timeline Integrity Side-by-Side (`micro_data_procedimento`, `cong_em_data`, `descong_em_data_transferencia`)
- **Part 8**: Treatment Outcomes & Non-Transfer Reasons Side-by-Side (`trat1_resultado_tratamento`, `trat1_motivo_nao_transferir`)


In [1]:
import duckdb
import pandas as pd
import numpy as np
from pyathena import connect
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
DUCKDB_PATH = '../../database/huntington_data_lake.duckdb'
ATHENA_REGION = 'sa-east-1'
ATHENA_WORKGROUP = 'datalake-admins'
ATHENA_SCHEMA = 'gold_huntington_prod'
STAGING_SCHEMA = 'gold_huntington_staging'

print("Connecting to DuckDB...")
duck_con = duckdb.connect(DUCKDB_PATH, read_only=True)

print(f"Connecting to AWS Athena ({ATHENA_SCHEMA})...")
try:
    ath_con = connect(
        region_name=ATHENA_REGION,
        work_group=ATHENA_WORKGROUP,
        schema_name=ATHENA_SCHEMA
    )
    ath_cur = ath_con.cursor()
    print("Successfully connected to AWS Athena.")
except Exception as e:
    print(f"Warning: Could not connect to Athena: {e}")
    ath_cur = None


Connecting to DuckDB...
Connecting to AWS Athena (gold_huntington_prod)...
Successfully connected to AWS Athena.


## 🔍 Part 1: DuckDB Queries (Local Gold Layer)

Executing local validation queries on `gold.clinisys_embrioes` and `gold.patient_info` in DuckDB.


In [2]:
# 1. Overall Totals (DuckDB)
query_duck_overall = '''
SELECT 
    'TOTAL' as grouping_type,
    'ALL' as grouping_value,
    COUNT(*) as total_rows,
    COUNT(DISTINCT oocito_id) as unique_oocito_id,
    COUNT(*) - COUNT(DISTINCT oocito_id) as duplicate_oocito_id_count,
    ROUND((COUNT(*) - COUNT(DISTINCT oocito_id)) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_duplicate_oocito_id,
    COUNT(trat1_id) as count_trat1_id,
    ROUND(COUNT(trat1_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat1_id,
    COUNT(emb_cong_id) as count_emb_cong_id,
    ROUND(COUNT(emb_cong_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_emb_cong_id,
    COUNT(cong_em_id) as count_cong_em_id,
    ROUND(COUNT(cong_em_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_cong_em_id,
    COUNT(descong_em_id) as count_descong_em_id,
    ROUND(COUNT(descong_em_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_descong_em_id,
    COUNT(trat2_id) as count_trat2_id,
    ROUND(COUNT(trat2_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat2_id
FROM gold.clinisys_embrioes
'''
df_duck_overall = duck_con.execute(query_duck_overall).df()
print("--- DuckDB Overall Totals ---")
display(df_duck_overall)

# 2. Per Year Breakdown (DuckDB)
query_duck_year = '''
SELECT 
    YEAR(CAST(micro_data_procedimento AS DATE)) as procedimento_year,
    COUNT(*) as total_rows,
    COUNT(DISTINCT oocito_id) as unique_oocito_id,
    COUNT(*) - COUNT(DISTINCT oocito_id) as duplicate_oocito_id_count,
    COUNT(trat1_id) as count_trat1_id,
    ROUND(COUNT(trat1_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat1_id,
    COUNT(emb_cong_id) as count_emb_cong_id,
    ROUND(COUNT(emb_cong_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_emb_cong_id,
    COUNT(cong_em_id) as count_cong_em_id,
    ROUND(COUNT(cong_em_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_cong_em_id,
    COUNT(descong_em_id) as count_descong_em_id,
    ROUND(COUNT(descong_em_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_descong_em_id,
    COUNT(trat2_id) as count_trat2_id,
    ROUND(COUNT(trat2_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat2_id
FROM gold.clinisys_embrioes
WHERE micro_data_procedimento IS NOT NULL
GROUP BY YEAR(CAST(micro_data_procedimento AS DATE))
ORDER BY procedimento_year DESC
'''
df_duck_year = duck_con.execute(query_duck_year).df()
print("\n--- DuckDB Per Year Breakdown ---")
display(df_duck_year)

# 3. Per Unidade Breakdown (DuckDB)
query_duck_unidade = '''
SELECT 
    COALESCE(p.unidade_nome, 'NA / Desconhecido') as unidade_nome,
    COUNT(*) as total_rows,
    COUNT(DISTINCT c.oocito_id) as unique_oocito_id,
    COUNT(*) - COUNT(DISTINCT c.oocito_id) as duplicate_oocito_id_count,
    COUNT(c.trat1_id) as count_trat1_id,
    ROUND(COUNT(c.trat1_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat1_id,
    COUNT(c.emb_cong_id) as count_emb_cong_id,
    ROUND(COUNT(c.emb_cong_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_emb_cong_id,
    COUNT(c.cong_em_id) as count_cong_em_id,
    ROUND(COUNT(c.cong_em_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_cong_em_id,
    COUNT(c.descong_em_id) as count_descong_em_id,
    ROUND(COUNT(c.descong_em_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_descong_em_id,
    COUNT(c.trat2_id) as count_trat2_id,
    ROUND(COUNT(c.trat2_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat2_id
FROM gold.clinisys_embrioes c
LEFT JOIN gold.patient_info p ON c.micro_prontuario = p.prontuario
GROUP BY COALESCE(p.unidade_nome, 'NA / Desconhecido')
ORDER BY total_rows DESC
'''
df_duck_unidade = duck_con.execute(query_duck_unidade).df()
print("\n--- DuckDB Per Unidade Breakdown ---")
display(df_duck_unidade)


--- DuckDB Overall Totals ---


,grouping_type,grouping_value,total_rows,unique_oocito_id,duplicate_oocito_id_count,pct_duplicate_oocito_id,count_trat1_id,pct_trat1_id,count_emb_cong_id,pct_emb_cong_id,count_cong_em_id,pct_cong_em_id,count_descong_em_id,pct_descong_em_id,count_trat2_id,pct_trat2_id
0,TOTAL,ALL,316427,316427,0,0.0,201942,63.82,56207,17.76,56207,17.76,15412,4.87,11105,3.51



--- DuckDB Per Year Breakdown ---


,procedimento_year,total_rows,unique_oocito_id,duplicate_oocito_id_count,count_trat1_id,pct_trat1_id,count_emb_cong_id,pct_emb_cong_id,count_cong_em_id,pct_cong_em_id,count_descong_em_id,pct_descong_em_id,count_trat2_id,pct_trat2_id
0,2026,19619,19619,0,17081,87.06,5962,30.39,5962,30.39,583,2.97,557,2.84
1,2025,36184,36184,0,29809,82.38,11538,31.89,11538,31.89,2275,6.29,2060,5.69
2,2024,34105,34105,0,21063,61.76,10435,30.60,10435,30.60,2625,7.70,2023,5.93
3,2023,34694,34694,0,19151,55.20,10774,31.05,10774,31.05,3277,9.45,2219,6.40
4,2022,31112,31112,0,13799,44.35,8152,26.20,8152,26.20,2707,8.70,1551,4.99
5,2021,18456,18456,0,2051,11.11,4592,24.88,4592,24.88,1393,7.55,391,2.12
6,2020,1738,1738,0,5,0.29,388,22.32,388,22.32,114,6.56,14,0.81
7,2019,106,106,0,0,0.00,31,29.25,31,29.25,11,10.38,0,0.00
8,2017,22,22,0,0,0.00,10,45.45,10,45.45,1,4.55,0,0.00
9,2016,35,35,0,0,0.00,3,8.57,3,8.57,1,2.86,0,0.00



--- DuckDB Per Unidade Breakdown ---


,unidade_nome,total_rows,unique_oocito_id,duplicate_oocito_id_count,count_trat1_id,pct_trat1_id,count_emb_cong_id,pct_emb_cong_id,count_cong_em_id,pct_cong_em_id,count_descong_em_id,pct_descong_em_id,count_trat2_id,pct_trat2_id
0,1. HTT SP - Ibirapuera,66363,66363,0,41543,62.60,16049,24.18,16049,24.18,3490,5.26,2419,3.65
1,5. HTT Belo Horizonte,65375,65375,0,61218,93.64,11552,17.67,11552,17.67,4933,7.55,4836,7.40
2,2. HTT SP - Vila Mariana,58278,58278,0,23542,40.40,9690,16.63,9690,16.63,2266,3.89,1109,1.90
3,3. HTT SP - ProFIV,34203,34203,0,9231,26.99,6391,18.69,6391,18.69,1690,4.94,524,1.53
4,10. HTT SP - DOE,28997,28997,0,24401,84.15,13,0.04,13,0.04,2,0.01,2,0.01
5,6. HTT Brasília,14307,14307,0,8124,56.78,3341,23.35,3341,23.35,787,5.50,546,3.82
6,NA / Desconhecido,14243,14243,0,11437,80.30,2068,14.52,2068,14.52,152,1.07,144,1.01
7,8. Unidade Salvador,12694,12694,0,10392,81.87,2848,22.44,2848,22.44,812,6.40,650,5.12
8,Não informado,10181,10181,0,2660,26.13,1554,15.26,1554,15.26,404,3.97,127,1.25
9,4. HTT SP - Campinas,8354,8354,0,6523,78.08,1897,22.71,1897,22.71,581,6.95,490,5.87


## ☁️ Part 2: AWS Athena Queries (Production Gold Layer)

Executing queries against AWS Athena (`{ATHENA_SCHEMA}.clinisys_embrioes`).


In [3]:
# Athena Queries Execution
if ath_cur is not None:
    # 1. Overall Totals (Athena)
    query_ath_overall = f'''
    SELECT 
        'TOTAL' as grouping_type,
        'ALL' as grouping_value,
        COUNT(*) as total_rows,
        COUNT(DISTINCT oocito_id) as unique_oocito_id,
        COUNT(*) - COUNT(DISTINCT oocito_id) as duplicate_oocito_id_count,
        ROUND(CAST(COUNT(*) - COUNT(DISTINCT oocito_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_duplicate_oocito_id,
        COUNT(trat1_id) as count_trat1_id,
        ROUND(CAST(COUNT(trat1_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat1_id,
        COUNT(emb_cong_id) as count_emb_cong_id,
        ROUND(CAST(COUNT(emb_cong_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_emb_cong_id,
        COUNT(cong_em_id) as count_cong_em_id,
        ROUND(CAST(COUNT(cong_em_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_cong_em_id,
        COUNT(descong_em_id) as count_descong_em_id,
        ROUND(CAST(COUNT(descong_em_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_descong_em_id,
        COUNT(trat2_id) as count_trat2_id,
        ROUND(CAST(COUNT(trat2_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat2_id
    FROM {ATHENA_SCHEMA}.clinisys_embrioes
    '''
    try:
        ath_cur.execute(query_ath_overall)
        df_ath_overall = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
        print("--- AWS Athena Overall Totals ---")
        display(df_ath_overall)
    except Exception as e:
        print(f"Error running Athena overall query: {e}")
        df_ath_overall = pd.DataFrame()

    # 2. Per Year Breakdown (Athena)
    query_ath_year = f'''
    SELECT 
        YEAR(CAST(micro_data_procedimento AS DATE)) as procedimento_year,
        COUNT(*) as total_rows,
        COUNT(DISTINCT oocito_id) as unique_oocito_id,
        COUNT(*) - COUNT(DISTINCT oocito_id) as duplicate_oocito_id_count,
        COUNT(trat1_id) as count_trat1_id,
        ROUND(CAST(COUNT(trat1_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat1_id,
        COUNT(emb_cong_id) as count_emb_cong_id,
        ROUND(CAST(COUNT(emb_cong_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_emb_cong_id,
        COUNT(cong_em_id) as count_cong_em_id,
        ROUND(CAST(COUNT(cong_em_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_cong_em_id,
        COUNT(descong_em_id) as count_descong_em_id,
        ROUND(CAST(COUNT(descong_em_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_descong_em_id,
        COUNT(trat2_id) as count_trat2_id,
        ROUND(CAST(COUNT(trat2_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat2_id
    FROM {ATHENA_SCHEMA}.clinisys_embrioes
    WHERE micro_data_procedimento IS NOT NULL
    GROUP BY YEAR(CAST(micro_data_procedimento AS DATE))
    ORDER BY procedimento_year DESC
    '''
    try:
        ath_cur.execute(query_ath_year)
        df_ath_year = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
        print("\n--- AWS Athena Per Year Breakdown ---")
        display(df_ath_year)
    except Exception as e:
        print(f"Error running Athena year query: {e}")
        df_ath_year = pd.DataFrame()

    # 3. Per Unidade Breakdown (Athena - joining with gold_huntington_staging.patient_info)
    query_ath_unidade = f'''
    SELECT 
        COALESCE(p.unidade_nome, 'NA / Desconhecido') as unidade_nome,
        COUNT(*) as total_rows,
        COUNT(DISTINCT c.oocito_id) as unique_oocito_id,
        COUNT(*) - COUNT(DISTINCT c.oocito_id) as duplicate_oocito_id_count,
        COUNT(c.trat1_id) as count_trat1_id,
        ROUND(CAST(COUNT(c.trat1_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat1_id,
        COUNT(c.emb_cong_id) as count_emb_cong_id,
        ROUND(CAST(COUNT(c.emb_cong_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_emb_cong_id,
        COUNT(c.cong_em_id) as count_cong_em_id,
        ROUND(CAST(COUNT(c.cong_em_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_cong_em_id,
        COUNT(c.descong_em_id) as count_descong_em_id,
        ROUND(CAST(COUNT(c.descong_em_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_descong_em_id,
        COUNT(c.trat2_id) as count_trat2_id,
        ROUND(CAST(COUNT(c.trat2_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat2_id
    FROM {ATHENA_SCHEMA}.clinisys_embrioes c
    LEFT JOIN {STAGING_SCHEMA}.patient_info p ON c.micro_prontuario = p.prontuario
    GROUP BY COALESCE(p.unidade_nome, 'NA / Desconhecido')
    ORDER BY total_rows DESC
    '''
    try:
        ath_cur.execute(query_ath_unidade)
        df_ath_unidade = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
        print("\n--- AWS Athena Per Unidade Breakdown ---")
        display(df_ath_unidade)
    except Exception as e:
        print(f"Error running Athena unidade query: {e}")
        df_ath_unidade = pd.DataFrame()
else:
    print("Athena cursor unavailable. Skipping Athena queries.")
    df_ath_overall = pd.DataFrame()
    df_ath_year = pd.DataFrame()
    df_ath_unidade = pd.DataFrame()


--- AWS Athena Overall Totals ---


,grouping_type,grouping_value,total_rows,unique_oocito_id,duplicate_oocito_id_count,pct_duplicate_oocito_id,count_trat1_id,pct_trat1_id,count_emb_cong_id,pct_emb_cong_id,count_cong_em_id,pct_cong_em_id,count_descong_em_id,pct_descong_em_id,count_trat2_id,pct_trat2_id
0,TOTAL,ALL,316580,316580,0,0.0,202111,63.84,56257,17.77,56257,17.77,15421,4.87,11114,3.51



--- AWS Athena Per Year Breakdown ---


,procedimento_year,total_rows,unique_oocito_id,duplicate_oocito_id_count,count_trat1_id,pct_trat1_id,count_emb_cong_id,pct_emb_cong_id,count_cong_em_id,pct_cong_em_id,count_descong_em_id,pct_descong_em_id,count_trat2_id,pct_trat2_id
0,7202,8,8,0,0,0.00,0,0.00,0,0.00,0,0.00,0,0.00
1,3036,9,9,0,9,100.00,1,11.11,1,11.11,0,0.00,0,0.00
2,3024,5,5,0,0,0.00,1,20.00,1,20.00,0,0.00,0,0.00
3,2202,2,2,0,2,100.00,1,50.00,1,50.00,1,50.00,1,50.00
4,2028,10,10,0,10,100.00,4,40.00,4,40.00,2,20.00,2,20.00
5,2026,19712,19712,0,17167,87.09,6024,30.56,6024,30.56,588,2.98,561,2.85
6,2025,36184,36184,0,29809,82.38,11538,31.89,11538,31.89,2279,6.30,2064,5.70
7,2024,34105,34105,0,21057,61.74,10435,30.60,10435,30.60,2625,7.70,2023,5.93
8,2023,34694,34694,0,19151,55.20,10774,31.05,10774,31.05,3277,9.45,2219,6.40
9,2022,31112,31112,0,13799,44.35,8152,26.20,8152,26.20,2709,8.71,1553,4.99



--- AWS Athena Per Unidade Breakdown ---


,unidade_nome,total_rows,unique_oocito_id,duplicate_oocito_id_count,count_trat1_id,pct_trat1_id,count_emb_cong_id,pct_emb_cong_id,count_cong_em_id,pct_cong_em_id,count_descong_em_id,pct_descong_em_id,count_trat2_id,pct_trat2_id
0,1. HTT SP - Ibirapuera,70632,70632,0,45231,64.04,16890,23.91,16890,23.91,3587,5.08,2503,3.54
1,6. HTT Belo Horizonte,67321,67321,0,63147,93.80,11810,17.54,11810,17.54,4960,7.37,4865,7.23
2,2. HTT SP - Vila Mariana,62809,62809,0,27314,43.49,10436,16.62,10436,16.62,2308,3.67,1138,1.81
3,3. HTT SP - ProFIV,34768,34768,0,9469,27.23,6391,18.38,6391,18.38,1679,4.83,526,1.51
4,5. HTT SP - DOE,29693,29693,0,25081,84.47,24,0.08,24,0.08,3,0.01,3,0.01
5,8. HTT Salvador,15927,15927,0,12927,81.16,3586,22.52,3586,22.52,1088,6.83,884,5.55
6,7. HTT Brasília,15099,15099,0,8743,57.90,3489,23.11,3489,23.11,803,5.32,560,3.71
7,Não informado,9774,9774,0,2345,23.99,1465,14.99,1465,14.99,379,3.88,112,1.15
8,4. HTT SP - Campinas,8724,8724,0,6840,78.40,1992,22.83,1992,22.83,584,6.69,493,5.65
9,NA / Desconhecido,746,746,0,35,4.69,0,0.00,0,0.00,0,0.00,0,0.00


## ⚖️ Part 3: Side-by-Side Reconciliation & Delta Comparison

Compares DuckDB vs Athena totals and highlights differences.


In [4]:
# Build Reconciliation Summary
if not df_duck_overall.empty:
    duck_summary = df_duck_overall.iloc[0].to_dict()
    
    if not df_ath_overall.empty:
        ath_summary = df_ath_overall.iloc[0].to_dict()
    else:
        ath_summary = {k: None for k in duck_summary.keys()}
        
    metrics = [
        'total_rows',
        'unique_oocito_id',
        'duplicate_oocito_id_count',
        'count_trat1_id',
        'count_emb_cong_id',
        'count_cong_em_id',
        'count_descong_em_id',
        'count_trat2_id'
    ]
    
    recon_rows = []
    for m in metrics:
        v_duck = duck_summary.get(m, 0)
        v_ath = ath_summary.get(m, 0) if ath_summary.get(m) is not None else 0
        diff = (v_duck - v_ath) if (v_duck is not None and v_ath is not None) else None
        pct_match = (100.0 * min(v_duck, v_ath) / max(v_duck, v_ath)) if (v_duck and v_ath) else (100.0 if v_duck == v_ath else 0.0)
        
        recon_rows.append({
            'Metric': m,
            'DuckDB (Local)': v_duck,
            'AWS Athena (Prod)': v_ath,
            'Delta (Duck - Ath)': diff,
            'Match %': f"{pct_match:.2f}%" if pct_match is not None else 'N/A'
        })
        
    df_recon = pd.DataFrame(recon_rows)
    print("=========================================================================")
    print("               SUMMARY RECONCILIATION: DUCKDB VS ATHENA                  ")
    print("=========================================================================")
    display(df_recon)


               SUMMARY RECONCILIATION: DUCKDB VS ATHENA                  


,Metric,DuckDB (Local),AWS Athena (Prod),Delta (Duck - Ath),Match %
0,total_rows,316427,316580,-153,99.95%
1,unique_oocito_id,316427,316580,-153,99.95%
2,duplicate_oocito_id_count,0,0,0,100.00%
3,count_trat1_id,201942,202111,-169,99.92%
4,count_emb_cong_id,56207,56257,-50,99.91%
5,count_cong_em_id,56207,56257,-50,99.91%
6,count_descong_em_id,15412,15421,-9,99.94%
7,count_trat2_id,11105,11114,-9,99.92%


## 🔎 Part 4: Oocito Discrepancy Analysis (Missing Records)

Identifies `oocito_id`s that exist **exclusively in DuckDB** or **exclusively in AWS Athena**, displaying sample records for detailed investigation.


In [5]:
# Part 4: Discrepancy Samples
print("Fetching set of unique oocito_ids from DuckDB...")
duck_oocito_ids = set(duck_con.execute("SELECT DISTINCT oocito_id FROM gold.clinisys_embrioes WHERE oocito_id IS NOT NULL").df()['oocito_id'])
print(f"Total unique oocito_ids in DuckDB: {len(duck_oocito_ids):,}")

if ath_cur is not None:
    try:
        print(f"Fetching set of unique oocito_ids from Athena ({ATHENA_SCHEMA})...")
        ath_cur.execute(f"SELECT DISTINCT oocito_id FROM {ATHENA_SCHEMA}.clinisys_embrioes WHERE oocito_id IS NOT NULL")
        ath_oocito_ids = set(r[0] for r in ath_cur.fetchall())
        print(f"Total unique oocito_ids in Athena: {len(ath_oocito_ids):,}")
    except Exception as e:
        print(f"Error querying Athena for oocito_ids: {e}")
        ath_oocito_ids = set()
else:
    print("Athena connection unavailable. Skipping Athena oocito set fetching.")
    ath_oocito_ids = set()

# Set difference logic
only_in_duck = list(duck_oocito_ids - ath_oocito_ids) if ath_oocito_ids else []
only_in_ath = list(ath_oocito_ids - duck_oocito_ids) if ath_oocito_ids else []

print("\n=========================================================================")
print(f"               DISCREPANCY SUMMARY: OOCITO_ID SET DIFFERENCE             ")
print("=========================================================================")
print(f"1. Oocito IDs present ONLY in DuckDB: {len(only_in_duck):,}")
print(f"2. Oocito IDs present ONLY in Athena: {len(only_in_ath):,}")

# Sample 1: Oocitos ONLY in DuckDB
if only_in_duck:
    sample_duck_ids = tuple(only_in_duck[:100])
    in_clause = f"({sample_duck_ids[0]})" if len(sample_duck_ids) == 1 else str(sample_duck_ids)
    
    query_sample_duck = f'''
    SELECT 
        c.oocito_id, 
        c.micro_prontuario, 
        c.micro_data_procedimento, 
        c.trat1_id, 
        c.emb_cong_id, 
        c.descong_em_id, 
        c.nome_medico,
        p.unidade_nome
    FROM gold.clinisys_embrioes c
    LEFT JOIN gold.patient_info p ON c.micro_prontuario = p.prontuario
    WHERE c.oocito_id IN {in_clause}
    LIMIT 15
    '''
    df_sample_duck = duck_con.execute(query_sample_duck).df()
    print("\n--- Sample Oocitos ONLY in DuckDB (Local) ---")
    display(df_sample_duck)
else:
    print("\nNo oocitos found exclusively in DuckDB (or Athena set not fetched).")

# Sample 2: Oocitos ONLY in Athena
if only_in_ath and ath_cur is not None:
    sample_ath_ids = tuple(only_in_ath[:100])
    in_clause_ath = f"({sample_ath_ids[0]})" if len(sample_ath_ids) == 1 else str(sample_ath_ids)
    
    query_sample_ath = f'''
    SELECT 
        c.oocito_id, 
        c.micro_prontuario, 
        c.micro_data_procedimento, 
        c.trat1_id, 
        c.emb_cong_id, 
        c.descong_em_id, 
        c.nome_medico,
        p.unidade_nome
    FROM {ATHENA_SCHEMA}.clinisys_embrioes c
    LEFT JOIN {STAGING_SCHEMA}.patient_info p ON c.micro_prontuario = p.prontuario
    WHERE c.oocito_id IN {in_clause_ath}
    LIMIT 15
    '''
    try:
        ath_cur.execute(query_sample_ath)
        df_sample_ath = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
        print("\n--- Sample Oocitos ONLY in AWS Athena (Prod) ---")
        display(df_sample_ath)
    except Exception as e:
        print(f"Error querying sample oocitos from Athena: {e}")
else:
    print("\nNo oocitos found exclusively in AWS Athena (or Athena set not fetched).")


Fetching set of unique oocito_ids from DuckDB...
Total unique oocito_ids in DuckDB: 316,427
Fetching set of unique oocito_ids from Athena (gold_huntington_prod)...
Total unique oocito_ids in Athena: 316,580

               DISCREPANCY SUMMARY: OOCITO_ID SET DIFFERENCE             
1. Oocito IDs present ONLY in DuckDB: 10
2. Oocito IDs present ONLY in Athena: 163

--- Sample Oocitos ONLY in DuckDB (Local) ---


,oocito_id,micro_prontuario,micro_data_procedimento,trat1_id,emb_cong_id,descong_em_id,nome_medico,unidade_nome
0,296038,887847,2026-01-26,40791,<NA>,<NA>,Sofia Andrade de Oliveira,8. Unidade Salvador
1,296039,887847,2026-01-26,40791,<NA>,<NA>,Sofia Andrade de Oliveira,8. Unidade Salvador
2,296040,887847,2026-01-26,40791,<NA>,<NA>,Sofia Andrade de Oliveira,8. Unidade Salvador
3,296041,887847,2026-01-26,40791,94960,<NA>,Sofia Andrade de Oliveira,8. Unidade Salvador
4,296042,887847,2026-01-26,40791,94961,<NA>,Sofia Andrade de Oliveira,8. Unidade Salvador
5,296043,887847,2026-01-26,40791,<NA>,<NA>,Sofia Andrade de Oliveira,8. Unidade Salvador
6,296044,887847,2026-01-26,40791,<NA>,<NA>,Sofia Andrade de Oliveira,8. Unidade Salvador
7,296045,887847,2026-01-26,40791,94980,21100,Sofia Andrade de Oliveira,8. Unidade Salvador
8,296046,887847,2026-01-26,40791,<NA>,<NA>,Sofia Andrade de Oliveira,8. Unidade Salvador
9,296047,887847,2026-01-26,40791,<NA>,<NA>,Sofia Andrade de Oliveira,8. Unidade Salvador



--- Sample Oocitos ONLY in AWS Athena (Prod) ---


,oocito_id,micro_prontuario,micro_data_procedimento,trat1_id,emb_cong_id,descong_em_id,nome_medico,unidade_nome
0,325994,916508,None,None,None,None,Mauricio Chehin,2. HTT SP - Vila Mariana
1,325995,916508,None,None,None,None,Mauricio Chehin,2. HTT SP - Vila Mariana
2,325996,916508,None,None,None,None,Mauricio Chehin,2. HTT SP - Vila Mariana
3,325997,916508,None,None,None,None,Mauricio Chehin,2. HTT SP - Vila Mariana
4,325998,916508,None,None,None,None,Mauricio Chehin,2. HTT SP - Vila Mariana
5,325999,916508,None,None,None,None,Mauricio Chehin,2. HTT SP - Vila Mariana
6,326000,916508,None,None,None,None,Mauricio Chehin,2. HTT SP - Vila Mariana
7,326001,916508,None,None,None,None,Mauricio Chehin,2. HTT SP - Vila Mariana
8,326002,916508,None,None,None,None,Mauricio Chehin,2. HTT SP - Vila Mariana
9,326003,916508,None,None,None,None,Mauricio Chehin,2. HTT SP - Vila Mariana


## 🔀 Part 5: Attribute Discrepancy Analysis (Per Key Attribute)

Compares matching `oocito_id`s across DuckDB and Athena to find records where specific key attributes differ.
Key attributes evaluated:
1. `trat1_id`
2. `emb_cong_id`
3. `cong_em_id`
4. `descong_em_id`
5. `trat2_id`
6. `oocito_embryo_number`


In [6]:
# Part 5 Data Preparation: Fetch matching oocito records for attribute comparison
print("Fetching key attribute columns from DuckDB...")
df_keys_duck = duck_con.execute('''
    SELECT 
        oocito_id, 
        micro_prontuario,
        micro_data_procedimento,
        nome_medico,
        trat1_id, 
        emb_cong_id, 
        cong_em_id, 
        descong_em_id, 
        trat2_id, 
        oocito_embryo_number
    FROM gold.clinisys_embrioes
    WHERE oocito_id IS NOT NULL
''').df()

if ath_cur is not None:
    try:
        print(f"Fetching key attribute columns from Athena ({ATHENA_SCHEMA})...")
        ath_cur.execute(f'''
            SELECT 
                oocito_id, 
                micro_prontuario,
                micro_data_procedimento,
                nome_medico,
                trat1_id, 
                emb_cong_id, 
                cong_em_id, 
                descong_em_id, 
                trat2_id, 
                oocito_embryo_number
            FROM {ATHENA_SCHEMA}.clinisys_embrioes
            WHERE oocito_id IS NOT NULL
        ''')
        df_keys_ath = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
        print(f"Loaded {len(df_keys_duck):,} DuckDB rows and {len(df_keys_ath):,} Athena rows.")
    except Exception as e:
        print(f"Error fetching Athena key columns: {e}")
        df_keys_ath = pd.DataFrame()
else:
    print("Athena cursor unavailable. Skipping Athena key comparison.")
    df_keys_ath = pd.DataFrame()

# Deduplicate on oocito_id for clean 1-to-1 attribute comparison
df_duck_unique = df_keys_duck.drop_duplicates(subset=['oocito_id'])
df_ath_unique = df_keys_ath.drop_duplicates(subset=['oocito_id']) if not df_keys_ath.empty else pd.DataFrame()

if not df_duck_unique.empty and not df_ath_unique.empty:
    df_merged_keys = pd.merge(
        df_duck_unique, 
        df_ath_unique, 
        on='oocito_id', 
        suffixes=('_duck', '_ath')
    )
    print(f"Total matching oocito_ids for attribute comparison: {len(df_merged_keys):,}")
else:
    df_merged_keys = pd.DataFrame()


Fetching key attribute columns from DuckDB...


Fetching key attribute columns from Athena (gold_huntington_prod)...
Loaded 316,427 DuckDB rows and 316,580 Athena rows.
Total matching oocito_ids for attribute comparison: 316,417


### 🔑 Key Comparison: `trat1_id`

In [7]:
# Discrepancy Check for Key: trat1_id
if not df_merged_keys.empty:
    col_duck = 'trat1_id_duck'
    col_ath = 'trat1_id_ath'
    
    # Identify mismatch mask (values differ AND not both NaN)
    v_duck = df_merged_keys[col_duck]
    v_ath = df_merged_keys[col_ath]
    
    mismatch_mask = (v_duck.isna() != v_ath.isna()) | (v_duck.notna() & v_ath.notna() & (v_duck != v_ath))
    df_diff_trat1_id = df_merged_keys[mismatch_mask]
    
    n_diff = len(df_diff_trat1_id)
    pct_diff = (n_diff * 100.0 / len(df_merged_keys)) if len(df_merged_keys) > 0 else 0.0
    
    print(f"=== KEY ATTRIBUTE: trat1_id ===")
    print(f"Total oocitos evaluated: {len(df_merged_keys):,}")
    print(f"Total mismatches: {n_diff:,} ({pct_diff:.2f}%)")
    
    if n_diff > 0:
        display_cols = ['oocito_id', 'micro_prontuario_duck', 'micro_data_procedimento_duck', col_duck, col_ath, 'nome_medico_duck']
        df_sample = df_diff_trat1_id[display_cols].rename(columns={
            col_duck: f'trat1_id_duckdb',
            col_ath: f'trat1_id_athena',
            'micro_prontuario_duck': 'prontuario',
            'micro_data_procedimento_duck': 'data_procedimento',
            'nome_medico_duck': 'nome_medico'
        }).head(15)
        display(df_sample)
    else:
        print(f"✅ All matching oocitos have identical values for 'trat1_id' across DuckDB and Athena.")
else:
    print("Athena dataset unavailable for comparison.")


=== KEY ATTRIBUTE: trat1_id ===
Total oocitos evaluated: 316,417
Total mismatches: 121 (0.04%)


,oocito_id,prontuario,data_procedimento,trat1_id_duckdb,trat1_id_athena,nome_medico
174862,182949,842186,NaT,24807,NaN,Eduardo Leme Alves da Motta
174863,182950,842186,NaT,24807,NaN,Eduardo Leme Alves da Motta
174864,182951,842186,NaT,24807,NaN,Eduardo Leme Alves da Motta
174865,182952,842186,NaT,24807,NaN,Eduardo Leme Alves da Motta
174866,182953,842186,NaT,24807,NaN,Eduardo Leme Alves da Motta
191819,200186,842186,NaT,27124,NaN,Eduardo Leme Alves da Motta
191820,200187,842186,NaT,27124,NaN,Eduardo Leme Alves da Motta
191821,200188,842186,NaT,27124,NaN,Eduardo Leme Alves da Motta
191822,200189,842186,NaT,27124,NaN,Eduardo Leme Alves da Motta
224661,233231,842186,2024-12-18,31172,NaN,Eduardo Leme Alves da Motta


### 🔑 Key Comparison: `emb_cong_id`

In [ ]:
# Discrepancy Check for Key: emb_cong_id
if not df_merged_keys.empty:
    col_duck = 'emb_cong_id_duck'
    col_ath = 'emb_cong_id_ath'
    
    # Identify mismatch mask (values differ AND not both NaN)
    v_duck = df_merged_keys[col_duck]
    v_ath = df_merged_keys[col_ath]
    
    mismatch_mask = (v_duck.isna() != v_ath.isna()) | (v_duck.notna() & v_ath.notna() & (v_duck != v_ath))
    df_diff_emb_cong_id = df_merged_keys[mismatch_mask]
    
    n_diff = len(df_diff_emb_cong_id)
    pct_diff = (n_diff * 100.0 / len(df_merged_keys)) if len(df_merged_keys) > 0 else 0.0
    
    print(f"=== KEY ATTRIBUTE: emb_cong_id ===")
    print(f"Total oocitos evaluated: {len(df_merged_keys):,}")
    print(f"Total mismatches: {n_diff:,} ({pct_diff:.2f}%)")
    
    if n_diff > 0:
        display_cols = ['oocito_id', 'micro_prontuario_duck', 'micro_data_procedimento_duck', col_duck, col_ath, 'nome_medico_duck']
        df_sample = df_diff_emb_cong_id[display_cols].rename(columns={
            col_duck: f'emb_cong_id_duckdb',
            col_ath: f'emb_cong_id_athena',
            'micro_prontuario_duck': 'prontuario',
            'micro_data_procedimento_duck': 'data_procedimento',
            'nome_medico_duck': 'nome_medico'
        }).head(15)
        display(df_sample)
    else:
        print(f"✅ All matching oocitos have identical values for 'emb_cong_id' across DuckDB and Athena.")
else:
    print("Athena dataset unavailable for comparison.")


=== KEY ATTRIBUTE: emb_cong_id ===
Total oocitos evaluated: 316,417
Total mismatches: 50 (0.02%)


,oocito_id,prontuario,data_procedimento,emb_cong_id_duckdb,emb_cong_id_athena,nome_medico
314278,323769,155429,2026-07-23,<NA>,101309.0,Hanna Park
314279,323770,155429,2026-07-23,<NA>,101310.0,Hanna Park
314280,323771,155429,2026-07-23,<NA>,101311.0,Hanna Park
314281,323772,155429,2026-07-23,<NA>,101312.0,Hanna Park
314282,323773,155429,2026-07-23,<NA>,101313.0,Hanna Park
314283,323774,155429,2026-07-23,<NA>,101314.0,Hanna Park
314284,323775,155429,2026-07-23,<NA>,101315.0,Hanna Park
314285,323776,155429,2026-07-23,<NA>,101316.0,Hanna Park
314286,323777,155429,2026-07-23,<NA>,101317.0,Hanna Park
314287,323778,155429,2026-07-23,<NA>,101318.0,Hanna Park


### 🔑 Key Comparison: `cong_em_id`

In [9]:
# Discrepancy Check for Key: cong_em_id
if not df_merged_keys.empty:
    col_duck = 'cong_em_id_duck'
    col_ath = 'cong_em_id_ath'
    
    # Identify mismatch mask (values differ AND not both NaN)
    v_duck = df_merged_keys[col_duck]
    v_ath = df_merged_keys[col_ath]
    
    mismatch_mask = (v_duck.isna() != v_ath.isna()) | (v_duck.notna() & v_ath.notna() & (v_duck != v_ath))
    df_diff_cong_em_id = df_merged_keys[mismatch_mask]
    
    n_diff = len(df_diff_cong_em_id)
    pct_diff = (n_diff * 100.0 / len(df_merged_keys)) if len(df_merged_keys) > 0 else 0.0
    
    print(f"=== KEY ATTRIBUTE: cong_em_id ===")
    print(f"Total oocitos evaluated: {len(df_merged_keys):,}")
    print(f"Total mismatches: {n_diff:,} ({pct_diff:.2f}%)")
    
    if n_diff > 0:
        display_cols = ['oocito_id', 'micro_prontuario_duck', 'micro_data_procedimento_duck', col_duck, col_ath, 'nome_medico_duck']
        df_sample = df_diff_cong_em_id[display_cols].rename(columns={
            col_duck: f'cong_em_id_duckdb',
            col_ath: f'cong_em_id_athena',
            'micro_prontuario_duck': 'prontuario',
            'micro_data_procedimento_duck': 'data_procedimento',
            'nome_medico_duck': 'nome_medico'
        }).head(15)
        display(df_sample)
    else:
        print(f"✅ All matching oocitos have identical values for 'cong_em_id' across DuckDB and Athena.")
else:
    print("Athena dataset unavailable for comparison.")


=== KEY ATTRIBUTE: cong_em_id ===
Total oocitos evaluated: 316,417
Total mismatches: 50 (0.02%)


,oocito_id,prontuario,data_procedimento,cong_em_id_duckdb,cong_em_id_athena,nome_medico
314278,323769,155429,2026-07-23,<NA>,33251.0,Hanna Park
314279,323770,155429,2026-07-23,<NA>,33251.0,Hanna Park
314280,323771,155429,2026-07-23,<NA>,33251.0,Hanna Park
314281,323772,155429,2026-07-23,<NA>,33251.0,Hanna Park
314282,323773,155429,2026-07-23,<NA>,33251.0,Hanna Park
314283,323774,155429,2026-07-23,<NA>,33251.0,Hanna Park
314284,323775,155429,2026-07-23,<NA>,33251.0,Hanna Park
314285,323776,155429,2026-07-23,<NA>,33251.0,Hanna Park
314286,323777,155429,2026-07-23,<NA>,33251.0,Hanna Park
314287,323778,155429,2026-07-23,<NA>,33251.0,Hanna Park


### 🔑 Key Comparison: `descong_em_id`

In [10]:
# Discrepancy Check for Key: descong_em_id
if not df_merged_keys.empty:
    col_duck = 'descong_em_id_duck'
    col_ath = 'descong_em_id_ath'
    
    # Identify mismatch mask (values differ AND not both NaN)
    v_duck = df_merged_keys[col_duck]
    v_ath = df_merged_keys[col_ath]
    
    mismatch_mask = (v_duck.isna() != v_ath.isna()) | (v_duck.notna() & v_ath.notna() & (v_duck != v_ath))
    df_diff_descong_em_id = df_merged_keys[mismatch_mask]
    
    n_diff = len(df_diff_descong_em_id)
    pct_diff = (n_diff * 100.0 / len(df_merged_keys)) if len(df_merged_keys) > 0 else 0.0
    
    print(f"=== KEY ATTRIBUTE: descong_em_id ===")
    print(f"Total oocitos evaluated: {len(df_merged_keys):,}")
    print(f"Total mismatches: {n_diff:,} ({pct_diff:.2f}%)")
    
    if n_diff > 0:
        display_cols = ['oocito_id', 'micro_prontuario_duck', 'micro_data_procedimento_duck', col_duck, col_ath, 'nome_medico_duck']
        df_sample = df_diff_descong_em_id[display_cols].rename(columns={
            col_duck: f'descong_em_id_duckdb',
            col_ath: f'descong_em_id_athena',
            'micro_prontuario_duck': 'prontuario',
            'micro_data_procedimento_duck': 'data_procedimento',
            'nome_medico_duck': 'nome_medico'
        }).head(15)
        display(df_sample)
    else:
        print(f"✅ All matching oocitos have identical values for 'descong_em_id' across DuckDB and Athena.")
else:
    print("Athena dataset unavailable for comparison.")


=== KEY ATTRIBUTE: descong_em_id ===
Total oocitos evaluated: 316,417
Total mismatches: 10 (0.00%)


,oocito_id,prontuario,data_procedimento,descong_em_id_duckdb,descong_em_id_athena,nome_medico
89811,96223,184090,2022-07-29,<NA>,21737.0,Gustavo Teles
89829,96241,184090,2022-07-29,<NA>,21737.0,Gustavo Teles
254975,263769,684301,2025-07-11,<NA>,21721.0,Herica Cristina Mendonça
260732,269654,778766,2025-08-15,<NA>,21731.0,Pedro Paulo Bastos Filho
283726,292687,894626,2025-12-17,<NA>,21728.0,Thais Sanches Domingues
283727,292688,894626,2025-12-17,<NA>,21728.0,Thais Sanches Domingues
285148,294384,883965,2026-01-18,<NA>,21732.0,Matheus Teixeira Roque
295792,305106,732029,2026-03-23,<NA>,21735.0,Gabriella de Oliveira Ferreira
307871,317166,902761,2026-06-06,<NA>,21733.0,Claudia Gomes Padilla
311776,321043,880249,2026-07-02,<NA>,21734.0,Claudia Gomes Padilla


### 🔑 Key Comparison: `trat2_id`

In [11]:
# Discrepancy Check for Key: trat2_id
if not df_merged_keys.empty:
    col_duck = 'trat2_id_duck'
    col_ath = 'trat2_id_ath'
    
    # Identify mismatch mask (values differ AND not both NaN)
    v_duck = df_merged_keys[col_duck]
    v_ath = df_merged_keys[col_ath]
    
    mismatch_mask = (v_duck.isna() != v_ath.isna()) | (v_duck.notna() & v_ath.notna() & (v_duck != v_ath))
    df_diff_trat2_id = df_merged_keys[mismatch_mask]
    
    n_diff = len(df_diff_trat2_id)
    pct_diff = (n_diff * 100.0 / len(df_merged_keys)) if len(df_merged_keys) > 0 else 0.0
    
    print(f"=== KEY ATTRIBUTE: trat2_id ===")
    print(f"Total oocitos evaluated: {len(df_merged_keys):,}")
    print(f"Total mismatches: {n_diff:,} ({pct_diff:.2f}%)")
    
    if n_diff > 0:
        display_cols = ['oocito_id', 'micro_prontuario_duck', 'micro_data_procedimento_duck', col_duck, col_ath, 'nome_medico_duck']
        df_sample = df_diff_trat2_id[display_cols].rename(columns={
            col_duck: f'trat2_id_duckdb',
            col_ath: f'trat2_id_athena',
            'micro_prontuario_duck': 'prontuario',
            'micro_data_procedimento_duck': 'data_procedimento',
            'nome_medico_duck': 'nome_medico'
        }).head(15)
        display(df_sample)
    else:
        print(f"✅ All matching oocitos have identical values for 'trat2_id' across DuckDB and Athena.")
else:
    print("Athena dataset unavailable for comparison.")


=== KEY ATTRIBUTE: trat2_id ===
Total oocitos evaluated: 316,417
Total mismatches: 10 (0.00%)


,oocito_id,prontuario,data_procedimento,trat2_id_duckdb,trat2_id_athena,nome_medico
89811,96223,184090,2022-07-29,<NA>,45810.0,Gustavo Teles
89829,96241,184090,2022-07-29,<NA>,45810.0,Gustavo Teles
254975,263769,684301,2025-07-11,<NA>,40340.0,Herica Cristina Mendonça
260732,269654,778766,2025-08-15,<NA>,46253.0,Pedro Paulo Bastos Filho
283726,292687,894626,2025-12-17,<NA>,46019.0,Thais Sanches Domingues
283727,292688,894626,2025-12-17,<NA>,46019.0,Thais Sanches Domingues
285148,294384,883965,2026-01-18,<NA>,45860.0,Matheus Teixeira Roque
295792,305106,732029,2026-03-23,<NA>,45833.0,Gabriella de Oliveira Ferreira
307871,317166,902761,2026-06-06,<NA>,45811.0,Claudia Gomes Padilla
311776,321043,880249,2026-07-02,<NA>,45801.0,Claudia Gomes Padilla


### 🔑 Key Comparison: `oocito_embryo_number`

In [12]:
# Discrepancy Check for Key: oocito_embryo_number
if not df_merged_keys.empty:
    col_duck = 'oocito_embryo_number_duck'
    col_ath = 'oocito_embryo_number_ath'
    
    # Identify mismatch mask (values differ AND not both NaN)
    v_duck = df_merged_keys[col_duck]
    v_ath = df_merged_keys[col_ath]
    
    mismatch_mask = (v_duck.isna() != v_ath.isna()) | (v_duck.notna() & v_ath.notna() & (v_duck != v_ath))
    df_diff_oocito_embryo_number = df_merged_keys[mismatch_mask]
    
    n_diff = len(df_diff_oocito_embryo_number)
    pct_diff = (n_diff * 100.0 / len(df_merged_keys)) if len(df_merged_keys) > 0 else 0.0
    
    print(f"=== KEY ATTRIBUTE: oocito_embryo_number ===")
    print(f"Total oocitos evaluated: {len(df_merged_keys):,}")
    print(f"Total mismatches: {n_diff:,} ({pct_diff:.2f}%)")
    
    if n_diff > 0:
        display_cols = ['oocito_id', 'micro_prontuario_duck', 'micro_data_procedimento_duck', col_duck, col_ath, 'nome_medico_duck']
        df_sample = df_diff_oocito_embryo_number[display_cols].rename(columns={
            col_duck: f'oocito_embryo_number_duckdb',
            col_ath: f'oocito_embryo_number_athena',
            'micro_prontuario_duck': 'prontuario',
            'micro_data_procedimento_duck': 'data_procedimento',
            'nome_medico_duck': 'nome_medico'
        }).head(15)
        display(df_sample)
    else:
        print(f"✅ All matching oocitos have identical values for 'oocito_embryo_number' across DuckDB and Athena.")
else:
    print("Athena dataset unavailable for comparison.")


=== KEY ATTRIBUTE: oocito_embryo_number ===
Total oocitos evaluated: 316,417
Total mismatches: 0 (0.00%)
✅ All matching oocitos have identical values for 'oocito_embryo_number' across DuckDB and Athena.


## 🧬 Part 6: PGT / Genetic Test Results Side-by-Side Comparison

Compares PGT genetic testing outcome distributions (`oocito_resultado_pgd`) side-by-side.
*Note: Empty strings, spaces, and NULL values are normalized together into `'NULL / Sem Teste'`.*


In [13]:
# 1. Distribution in DuckDB (Normalized NULLs/empty)
query_pgt_duck = '''
SELECT 
    COALESCE(NULLIF(TRIM(oocito_ResultadoPGD), ''), 'NULL / Sem Teste') as resultado_pgt,
    COUNT(*) as count_duck
FROM gold.clinisys_embrioes
GROUP BY 1
'''
df_pgt_duck = duck_con.execute(query_pgt_duck).df()

# 2. Distribution in Athena (Normalized NULLs/empty)
if ath_cur is not None:
    query_pgt_ath = f'''
    SELECT 
        COALESCE(NULLIF(TRIM(oocito_resultado_pgd), ''), 'NULL / Sem Teste') as resultado_pgt,
        COUNT(*) as count_ath
    FROM {ATHENA_SCHEMA}.clinisys_embrioes
    GROUP BY 1
    '''
    try:
        ath_cur.execute(query_pgt_ath)
        df_pgt_ath = pd.DataFrame(ath_cur.fetchall(), columns=['resultado_pgt', 'count_ath'])
    except Exception as e:
        print(f"Error querying Athena PGT results: {e}")
        df_pgt_ath = pd.DataFrame()
else:
    df_pgt_ath = pd.DataFrame()

# 3. Side-by-Side Reconciliation Table
if not df_pgt_duck.empty and not df_pgt_ath.empty:
    df_pgt_side = pd.merge(df_pgt_duck, df_pgt_ath, on='resultado_pgt', how='outer').fillna(0)
    df_pgt_side['count_duck'] = df_pgt_side['count_duck'].astype(int)
    df_pgt_side['count_ath'] = df_pgt_side['count_ath'].astype(int)
    df_pgt_side['delta'] = df_pgt_side['count_duck'] - df_pgt_side['count_ath']
    df_pgt_side['match_pct'] = df_pgt_side.apply(
        lambda r: f"{(100.0 * min(r['count_duck'], r['count_ath']) / max(r['count_duck'], r['count_ath'])):.2f}%" if max(r['count_duck'], r['count_ath']) > 0 else '100.00%',
        axis=1
    )
    df_pgt_side = df_pgt_side.sort_values(by='count_duck', ascending=False).reset_index(drop=True)
    df_pgt_side = df_pgt_side.rename(columns={
        'resultado_pgt': 'PGT Result',
        'count_duck': 'DuckDB Count',
        'count_ath': 'Athena Count',
        'delta': 'Delta (Duck - Ath)',
        'match_pct': 'Match %'
    })
    print("=========================================================================")
    print("           PGT RESULT DISTRIBUTION: DUCKDB VS ATHENA SIDE-BY-SIDE         ")
    print("=========================================================================")
    display(df_pgt_side)


           PGT RESULT DISTRIBUTION: DUCKDB VS ATHENA SIDE-BY-SIDE         


,PGT Result,DuckDB Count,Athena Count,Delta (Duck - Ath),Match %
0,NULL / Sem Teste,285250,285356,-106,99.96%
1,Aneuploide,8403,8419,-16,99.81%
2,Alterado,7361,7361,0,100.00%
3,Euploide,6251,6261,-10,99.84%
4,Normal,4828,4828,0,100.00%
5,Não analisado,2389,2403,-14,99.42%
6,Aneuploide complexo,705,710,-5,99.30%
7,Mosaico baixo grau,377,378,-1,99.74%
8,Mosaico,335,335,0,100.00%
9,Em risco,159,159,0,100.00%


## 🗓️ Part 7: Date Sequence & Timeline Integrity Side-by-Side

Validates chronological integrity and compares date counts side-by-side across procedure dates (`micro_data_procedimento`), freezing dates (`cong_em_data`), and transfer dates (`descong_em_data_transferencia`).


In [14]:
# Date Sequence Validation
query_date_duck = '''
SELECT 
    COUNT(*) as total_rows,
    COUNT(micro_data_procedimento) as count_data_procedimento,
    COUNT(cong_em_Data) as count_data_congelamento,
    COUNT(descong_em_DataTransferencia) as count_data_transferencia,
    SUM(CASE WHEN CAST(cong_em_Data AS DATE) < CAST(micro_data_procedimento AS DATE) THEN 1 ELSE 0 END) as anomaly_freeze_before_proc,
    SUM(CASE WHEN CAST(descong_em_DataTransferencia AS DATE) < CAST(cong_em_Data AS DATE) THEN 1 ELSE 0 END) as anomaly_transfer_before_freeze
FROM gold.clinisys_embrioes
'''
dict_date_duck = duck_con.execute(query_date_duck).df().iloc[0].to_dict()

if ath_cur is not None:
    query_date_ath = f'''
    SELECT 
        COUNT(*) as total_rows,
        COUNT(micro_data_procedimento) as count_data_procedimento,
        COUNT(cong_em_data) as count_data_congelamento,
        COUNT(descong_em_data_transferencia) as count_data_transferencia,
        SUM(CASE WHEN CAST(cong_em_data AS DATE) < CAST(micro_data_procedimento AS DATE) THEN 1 ELSE 0 END) as anomaly_freeze_before_proc,
        SUM(CASE WHEN CAST(descong_em_data_transferencia AS DATE) < CAST(cong_em_data AS DATE) THEN 1 ELSE 0 END) as anomaly_transfer_before_freeze
    FROM {ATHENA_SCHEMA}.clinisys_embrioes
    '''
    try:
        ath_cur.execute(query_date_ath)
        dict_date_ath = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description]).iloc[0].to_dict()
    except Exception as e:
        print(f"Error running Athena date query: {e}")
        dict_date_ath = {}
else:
    dict_date_ath = {}

if dict_date_duck and dict_date_ath:
    date_side_rows = []
    for m in dict_date_duck.keys():
        v_duck = dict_date_duck[m]
        v_ath = dict_date_ath.get(m, 0)
        diff = v_duck - v_ath
        pct = (100.0 * min(v_duck, v_ath) / max(v_duck, v_ath)) if max(v_duck, v_ath) > 0 else 100.0
        date_side_rows.append({
            'Timeline Metric': m,
            'DuckDB (Local)': v_duck,
            'AWS Athena (Prod)': v_ath,
            'Delta (Duck - Ath)': diff,
            'Match %': f"{pct:.2f}%"
        })
    df_date_side = pd.DataFrame(date_side_rows)
    print("=========================================================================")
    print("        DATE SEQUENCE INTEGRITY & COUNTS: DUCKDB VS ATHENA SIDE-BY-SIDE   ")
    print("=========================================================================")
    display(df_date_side)


        DATE SEQUENCE INTEGRITY & COUNTS: DUCKDB VS ATHENA SIDE-BY-SIDE   


,Timeline Metric,DuckDB (Local),AWS Athena (Prod),Delta (Duck - Ath),Match %
0,total_rows,316427.0,316580,-153.0,99.95%
1,count_data_procedimento,176163.0,176290,-127.0,99.93%
2,count_data_congelamento,56177.0,56227,-50.0,99.91%
3,count_data_transferencia,15408.0,15420,-12.0,99.92%
4,anomaly_freeze_before_proc,136.0,155,-19.0,87.74%
5,anomaly_transfer_before_freeze,21.0,21,0.0,100.00%


## 🩺 Part 8: Treatment Outcomes & Cancellation Reasons Side-by-Side

Compares clinical outcome classifications (`trat1_resultado_tratamento`) and non-transfer reasons (`trat1_motivo_nao_transferir`) side-by-side between DuckDB and Athena.
*Note: Empty strings, spaces, and NULL values are normalized together into `'NULL / N/A'`.*


In [15]:
# 1. Outcomes Side-by-Side (Normalized NULLs/empty)
q_out_duck = '''
SELECT 
    COALESCE(NULLIF(TRIM(trat1_resultado_tratamento), ''), 'NULL / N/A') as resultado_tratamento,
    COUNT(*) as count_duck
FROM gold.clinisys_embrioes
GROUP BY 1
'''
df_out_duck = duck_con.execute(q_out_duck).df()

if ath_cur is not None:
    q_out_ath = f'''
    SELECT 
        COALESCE(NULLIF(TRIM(trat1_resultado_tratamento), ''), 'NULL / N/A') as resultado_tratamento,
        COUNT(*) as count_ath
    FROM {ATHENA_SCHEMA}.clinisys_embrioes
    GROUP BY 1
    '''
    try:
        ath_cur.execute(q_out_ath)
        df_out_ath = pd.DataFrame(ath_cur.fetchall(), columns=['resultado_tratamento', 'count_ath'])
    except Exception as e:
        print(f"Error querying Athena outcomes: {e}")
        df_out_ath = pd.DataFrame()
else:
    df_out_ath = pd.DataFrame()

if not df_out_duck.empty and not df_out_ath.empty:
    df_out_side = pd.merge(df_out_duck, df_out_ath, on='resultado_tratamento', how='outer').fillna(0)
    df_out_side['count_duck'] = df_out_side['count_duck'].astype(int)
    df_out_side['count_ath'] = df_out_side['count_ath'].astype(int)
    df_out_side['delta'] = df_out_side['count_duck'] - df_out_side['count_ath']
    df_out_side['match_pct'] = df_out_side.apply(
        lambda r: f"{(100.0 * min(r['count_duck'], r['count_ath']) / max(r['count_duck'], r['count_ath'])):.2f}%" if max(r['count_duck'], r['count_ath']) > 0 else '100.00%',
        axis=1
    )
    df_out_side = df_out_side.sort_values(by='count_duck', ascending=False).reset_index(drop=True)
    df_out_side = df_out_side.rename(columns={
        'resultado_tratamento': 'Treatment Outcome',
        'count_duck': 'DuckDB Count',
        'count_ath': 'Athena Count',
        'delta': 'Delta (Duck - Ath)',
        'match_pct': 'Match %'
    })
    print("=========================================================================")
    print("       TREATMENT OUTCOME DISTRIBUTION: DUCKDB VS ATHENA SIDE-BY-SIDE     ")
    print("=========================================================================")
    display(df_out_side)

# 2. Non-Transfer Reasons Side-by-Side (Normalized NULLs/empty)
q_cancel_duck = '''
SELECT 
    COALESCE(NULLIF(TRIM(trat1_motivo_nao_transferir), ''), 'NULL / Transferido ou N/A') as motivo_nao_transferir,
    COUNT(*) as count_duck
FROM gold.clinisys_embrioes
WHERE trat1_motivo_nao_transferir IS NOT NULL AND TRIM(trat1_motivo_nao_transferir) != ''
GROUP BY 1
'''
df_cancel_duck = duck_con.execute(q_cancel_duck).df()

if ath_cur is not None:
    q_cancel_ath = f'''
    SELECT 
        COALESCE(NULLIF(TRIM(trat1_motivo_nao_transferir), ''), 'NULL / Transferido ou N/A') as motivo_nao_transferir,
        COUNT(*) as count_ath
    FROM {ATHENA_SCHEMA}.clinisys_embrioes
    WHERE trat1_motivo_nao_transferir IS NOT NULL AND TRIM(trat1_motivo_nao_transferir) != ''
    GROUP BY 1
    '''
    try:
        ath_cur.execute(q_cancel_ath)
        df_cancel_ath = pd.DataFrame(ath_cur.fetchall(), columns=['motivo_nao_transferir', 'count_ath'])
    except Exception as e:
        print(f"Error querying Athena cancel reasons: {e}")
        df_cancel_ath = pd.DataFrame()
else:
    df_cancel_ath = pd.DataFrame()

if not df_cancel_duck.empty and not df_cancel_ath.empty:
    df_cancel_side = pd.merge(df_cancel_duck, df_cancel_ath, on='motivo_nao_transferir', how='outer').fillna(0)
    df_cancel_side['count_duck'] = df_cancel_side['count_duck'].astype(int)
    df_cancel_side['count_ath'] = df_cancel_side['count_ath'].astype(int)
    df_cancel_side['delta'] = df_cancel_side['count_duck'] - df_cancel_side['count_ath']
    df_cancel_side['match_pct'] = df_cancel_side.apply(
        lambda r: f"{(100.0 * min(r['count_duck'], r['count_ath']) / max(r['count_duck'], r['count_ath'])):.2f}%" if max(r['count_duck'], r['count_ath']) > 0 else '100.00%',
        axis=1
    )
    df_cancel_side = df_cancel_side.sort_values(by='count_duck', ascending=False).reset_index(drop=True)
    df_cancel_side = df_cancel_side.rename(columns={
        'motivo_nao_transferir': 'Non-Transfer Reason',
        'count_duck': 'DuckDB Count',
        'count_ath': 'Athena Count',
        'delta': 'Delta (Duck - Ath)',
        'match_pct': 'Match %'
    })
    print("\n=========================================================================")
    print("      NON-TRANSFER REASON DISTRIBUTION: DUCKDB VS ATHENA SIDE-BY-SIDE   ")
    print("=========================================================================")
    display(df_cancel_side)


       TREATMENT OUTCOME DISTRIBUTION: DUCKDB VS ATHENA SIDE-BY-SIDE     


,Treatment Outcome,DuckDB Count,Athena Count,Delta (Duck - Ath),Match %
0,NULL / N/A,150359,150304,55,99.96%
1,No transfer,116158,116264,-106,99.91%
2,Congelamento de Óvulos,36913,37018,-105,99.72%
3,Gestação Clínica,6058,6058,0,100.00%
4,Negativo,5010,5007,3,99.94%
5,Gestação Química Confirmada,985,985,0,100.00%
6,Gestação Química,683,683,0,100.00%
7,Biópsia embrionária,252,252,0,100.00%
8,Biópsia Realizada,9,9,0,100.00%



      NON-TRANSFER REASON DISTRIBUTION: DUCKDB VS ATHENA SIDE-BY-SIDE   


,Non-Transfer Reason,DuckDB Count,Athena Count,Delta (Duck - Ath),Match %
0,Criopreservação total,104266,104417,-151,99.86%
1,Doação,29575,29594,-19,99.94%
2,Congelamento total,16194,16194,0,100.00%
3,Ausência de embriões geneticamente normais,9765,9773,-8,99.92%
4,Ausência de desenvolvimento embrionário,5311,5313,-2,99.96%
5,Congelamento de Oócitos,3991,3991,0,100.00%
6,Outras complicações,944,944,0,100.00%
7,Doação terceira via,773,773,0,100.00%
8,Hiperestímulo,559,559,0,100.00%
9,Outros,425,425,0,100.00%


In [16]:
# Close database connections
print("Closing database connections...")
try:
    duck_con.close()
    print("DuckDB connection closed.")
except Exception as e:
    print(f"Error closing DuckDB connection: {e}")

try:
    ath_con.close()
    print("Athena connection closed.")
except Exception as e:
    print(f"Error closing Athena connection: {e}")

Closing database connections...
DuckDB connection closed.
Athena connection closed.
